# Reasoning-trace generation, independent-solve variant (teacher: Qwen3-Next-80B-A3B-Thinking)

**Differs from `qwen3-next-80b-conr-trace-gen.ipynb` in one fundamental way**: the teacher is
given ONLY `pre_text`/`table`/`post_text`/`question` -- no gold program, no gold answer -- and must
genuinely derive the program itself. The original notebook's CoNR prompt handed the teacher the
verified program/answer and asked it to write a trace that *rationalizes* that given program
(backward rationalization); that guarantees a ~98% exact-match yield but risks teaching the student
a stylistically-plausible-but-not-causally-real reasoning pattern, since the teacher never actually
had to search for the answer. This variant trades yield for genuineness: a sample is only kept if the
teacher's *independently derived* program matches gold exactly -- same validation logic
(`is_exact_program_match`) as before, just nothing is given away beforehand.

**Cost note**: because the teacher can now genuinely get samples wrong, expect the exact-match yield
to be well below the original's ~98%. `SAMPLE_LIMIT` below defaults to a small trial (30 samples) so
you can see the real yield/cost on a cheap run before committing more budget -- raise it (or set to
`None` for a full run) only after you've reviewed the trial's output file.

The CoNR prompt/parser/extraction logic here was validated on two 150-sample
trial runs first (see `conr-v1/conr_trace_trial_150.json` and
`conr_trace_trial_150_v2.json`): 92.0% -> 98.0% exact program match after
fixing a nested-parens parser bug, a `<think>`-tag extraction bug, and
widening `max_tokens`. The 3 remaining mismatches in the v2 trial were all
traced to actual errors in `train.json`'s gold labels (a malformed program
missing a closing paren, an invalid `7%` operand, and one gold program whose
division direction contradicts its own question wording) -- not the pipeline.

Run on my own server (2x RTX PRO 6000 96GB + 4 CPU / 16GB RAM). Upload `train.json` via file browser to `/root/data/train.json` before running (or adjust `TRAIN_JSON_PATH` below).

Output: one JSON file with the raw teacher output for all 2993 samples, plus a
validation flag per sample (program exact-match against gold, denylist check).
No mid-run checkpointing -- this runs as a single `llm.generate()` batch call,
same as the trial runs; a failure partway through loses the whole run and
needs a clean re-run from the top.

In [ ]:
import os
from pathlib import Path

# Point the HuggingFace cache at the attached Modal Volume (mounted at
# /mnt/qwen-cache) instead of the container's ephemeral local disk, so the
# ~160GB model download only ever happens once -- subsequent kernel restarts /
# container restarts load from the Volume instead of re-downloading. Must run
# before `from vllm import LLM, ...` below (env vars are read at import time).
# If your Volume is mounted at a different path, update this to match.
HF_CACHE = Path("/mnt/qwen-cache")
os.environ["HF_HUB_CACHE"] = str(HF_CACHE)

# Is the teacher already cached in the Volume, or will this run pay the
# 15-30 minute download again (billed, GPUs idle)? Check before committing.
if not HF_CACHE.exists():
    print(f"!! {HF_CACHE} does not exist -- no Volume attached at that path.")
    print("   Attach one via the notebook sidebar (filesystem tab), otherwise")
    print("   the model downloads to ephemeral disk and is lost on shutdown.")
else:
    hits = list(HF_CACHE.glob("**/models--Qwen--Qwen3-Next-80B-A3B-Thinking"))
    if not hits:
        print(f"{HF_CACHE} is attached but the teacher is NOT cached yet.")
        print("   -> this run will download ~160GB (15-30+ min, billed).")
    else:
        size_gb = sum(f.stat().st_size for f in hits[0].rglob("*") if f.is_file()) / 1024**3
        print(f"Teacher already cached in the Volume: {hits[0]}")
        print(f"   cached size: {size_gb:.1f} GB -> no re-download expected.")

In [ ]:
%%capture
!pip install -q vllm tabulate
!pip install -q -U "typing_extensions>=4.12" "pydantic>=2.9"
# IMPORTANT: restart the kernel after this cell finishes (Kernel > Restart),
# then re-run from the top. Server's base image ships an older
# typing_extensions that gets imported into the running process before this
# cell runs; upgrading the package on disk doesn't unload the old module
# already in memory, so vllm's `from pydantic import ...` chain still hits
# the stale one (ImportError: cannot import name 'Sentinel' from
# 'typing_extensions') unless the kernel is restarted.

In [ ]:
import json
from pathlib import Path

# Adjust if you uploaded train.json somewhere else via the Server Web UI file browser.
TRAIN_JSON_PATH = Path("/root/valid.json")
# Saved inside the attached Modal Volume (not /root), so results survive a
# kernel restart or container shutdown -- /root is ephemeral and is wiped
# when the container is torn down.
OUTPUT_DIR = Path("/mnt/qwen-cache/outputs/conr_trace_independent_solve")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not TRAIN_JSON_PATH.exists():
    raise FileNotFoundError(
        f"{TRAIN_JSON_PATH} not found. Upload train.json via the Server Web UI "
        "file browser first, or update TRAIN_JSON_PATH to match where you put it."
    )

train_data = json.load(open(TRAIN_JSON_PATH, encoding="utf-8"))
print(f"Loaded {len(train_data)} samples from {TRAIN_JSON_PATH}")

## Evidence-type classification (for reporting, not sampling)

Classifies each sample by which evidence type its gold program's numeric
arguments come from (table values vs. text values vs. both). Unlike the
150-sample trial, this run uses every sample in `train.json` -- the
classification is kept only so the per-category breakdown can be reported
alongside the overall exact-match rate.

In [ ]:
import re

def numbers_in(text):
    return set(re.findall(r"-?\d+\.?\d*", text))

def classify_evidence_type(sample):
    prog = sample["qa"]["program"]
    pre_text = " ".join(sample["pre_text"])
    post_text = " ".join(sample["post_text"])
    text_nums = numbers_in(pre_text) | numbers_in(post_text)
    table_flat = " ".join(" ".join(row) for row in sample["table"])
    table_nums = numbers_in(table_flat)
    prog_nums = set(re.findall(r"-?\d+\.?\d*", prog))

    uses_text = bool(prog_nums & text_nums)
    uses_table = bool(prog_nums & table_nums)

    if uses_table and not uses_text:
        return "table_only"
    elif uses_text and not uses_table:
        return "text_only"
    elif uses_text and uses_table:
        return "table_text"
    return "unclassified"

buckets = {"table_only": [], "table_text": [], "text_only": [], "unclassified": []}
for i, s in enumerate(train_data):
    buckets[classify_evidence_type(s)].append(i)

for k, v in buckets.items():
    print(f"{k}: {len(v)}")

# Full run. The 100-sample trial validated the v2 convention rules first:
# 65.0% exact match vs the v1 prompt's 14.4%, with rule violations at
# R1 0.0% / R2 0.0% / R3 1.0% / R5 0.0% and no missing tags or leaks.
# Set this back to a small int if the prompt is changed again.
SAMPLE_LIMIT = None
sample_idx = list(range(len(train_data)))
if SAMPLE_LIMIT is not None:
    sample_idx = sample_idx[:SAMPLE_LIMIT]
print(f"\nTotal samples to process: {len(sample_idx)} (SAMPLE_LIMIT={SAMPLE_LIMIT})")

## Context formatting (same as the 0-shot/1-shot/few-shot/SFT notebooks)

In [ ]:
from tabulate import tabulate

def formatting_pre_text(sample):
    return "\n".join(sample["pre_text"])

def formatting_table(sample):
    return tabulate(sample["table"][1:], headers=sample["table"][0], tablefmt="github")

def formatting_post_text(sample):
    return "\n".join(sample["post_text"])

samples = []
for i in sample_idx:
    s = train_data[i]
    samples.append({
        "train_index": i,
        "evidence_type": classify_evidence_type(s),
        "pre_text": formatting_pre_text(s),
        "table": formatting_table(s),
        "post_text": formatting_post_text(s),
        "question": s["qa"]["question"],
        "program": s["qa"]["program"],
        "answer": s["qa"]["exe_ans"],
    })

print(f"Prepared {len(samples)} samples for trace generation.")
samples[0]

## Independent-solve prompt (v2 -- convention-hardened)

The teacher gets only context + question, exactly what the student sees at
inference time. It must derive the program itself.

**v2 changes.** The v1 run reached only 14.4% exact match, but an error analysis
of all 2,993 attempts showed the failures were overwhelmingly *formatting*, not
reasoning: 37.6% nested calls inside arguments, 27.3% appended a
`multiply(#N, 100)` to turn a ratio into a percent, 9.3% wrapped a single value
in `table_sum(...)`, 9.0% invented operators such as `table_value(...)`, and 9.2%
never emitted a `<program>` at all (the model drifted into English self-talk and
ran out of tokens). Each rule below targets one measured failure mode, and the
conventions are stated as counts over the 2,993 gold programs so they are not
guesses:

- nested calls appear in **6 / 2993** gold programs -> effectively never
- `table_*` with a row label + `none` appears **454** times, with explicit
  numbers **2** times -> the row-label form is the convention
- gold keeps ratios as decimal fractions (`divide(180, 8012)`), never `* 100`

The prompt also now forbids writing the program inside the `<think>` prose,
because v1 traces that did so spelled out a *different* program than the one
being trained on.

In [ ]:
CONR_SYSTEM_PROMPT = """You are a senior financial analyst solving a numerical question about a
Vietnamese financial document. You are given only the context (text before a
table, the table itself, text after the table) and the question -- nothing
else. You must derive the answer yourself.

Generate a sequential computation program to answer the question, using ONLY
the following 10 operators:

1. add(a, b) -> a + b
2. subtract(a, b) -> a - b
3. multiply(a, b) -> a * b
4. divide(a, b) -> a / b
5. exp(a, b) -> a^b
6. greater(a, b) -> 1.0 if a > b, else 0.0
7. table_sum(val1, val2, val3, ...) -> sum of the values
8. table_average(val1, val2, val3, ...) -> arithmetic mean of the values
9. table_max(val1, val2, val3, ...) -> maximum of the values
10. table_min(val1, val2, val3, ...) -> minimum of the values

### PROGRAM FORMAT RULES (follow exactly -- these are the dataset's conventions):

R1. NEVER nest one operator inside another's arguments. Write each operation as
    its own step, separated by ", ", and refer to an earlier step's result with
    #0 (step 1), #1 (step 2), and so on.
      WRONG: divide(subtract(2438.4, 2408.8), 2408.8)
      RIGHT: subtract(2438.4, 2408.8), divide(#0, 2408.8)

R2. NEVER multiply by 100 to turn a ratio into a percentage. Even when the
    question asks "bao nhieu phan tram" / "ty le ... la bao nhieu", the answer
    stays a decimal fraction and the program ends at the division.
      WRONG: divide(180, 8012), multiply(#0, 100)
      RIGHT: divide(180, 8012)

R3. Use ONLY the 10 operators above. Do not invent operators such as
    table_value(...), percent(...), sum(...) or lookup(...).

R4. When the question aggregates an ENTIRE row of the table (its total, average,
    max or min across all periods), reference the row by its label exactly as it
    appears in the table's first column, with `none` as the second argument --
    do not enumerate the row's values.
      WRONG: table_max(1584, 1261, 5786, 3428, 1479, 2290)
      RIGHT: table_max(EPS (VND), none)

R5. Never wrap a single value in a table operator; write the number itself.
      WRONG: subtract(table_sum(17005), table_sum(12207))
      RIGHT: subtract(17005, 12207)

R6. When you combine a few specific values that you read off individually
    (rather than a whole row), chain binary operators instead of using a table
    operator.
      WRONG: table_sum(6851, 9091, 14606)
      RIGHT: add(6851, 9091), add(#0, 14606)

R7. Write numbers as plain digits: drop currency symbols and thousand
    separators, and keep the decimal point. "3.564 ty" -> 3564, "$ 620,125" ->
    620125, "108.50" -> 108.50. Keep a literal percent sign only when the
    context states the value as a percentage that enters the calculation
    directly, e.g. add(1, 15%). If a needed value is missing, use 'none'.

### OUTPUT FORMAT (strict):
<think>
[Your reasoning, IN VIETNAMESE, 3-6 sentences. Must:
 1. Identify which specific values are needed and WHERE they come from
    (quote the exact row/column label from the table, or the exact sentence
    from pre_text/post_text) -- never state a number without saying where it
    was found.
 2. Explain WHY each operator was chosen (e.g. "vi cau hoi yeu cau ty le giua
    hai ky nen ta chia gia tri nam sau cho nam truoc" rather than just naming
    the operator).
 3. If the calculation has multiple steps, walk through them in order,
    referring to intermediate results the way the program does (ket qua buoc 1,
    ket qua buoc 2, ...).]
</think>
<program>YOUR_DERIVED_PROGRAM</program>

### REASONING STYLE RULES:
S1. Write the reasoning in Vietnamese. Do not think out loud in English.
S2. Do NOT write the final program string (or any concrete operator call with
    real numbers, such as "divide(180, 8012)") inside the <think> block. The
    program belongs only in the <program> block. Naming an operator in prose
    (e.g. "ta dung phep divide") is fine.
S3. Commit to one derivation. Do not second-guess yourself, re-derive the
    calculation, or narrate doubts ("Wait", "Hmm", "Nhung ma", "Vay dung").
S4. Keep the <think> block to 3-6 sentences and stop. Do not pad with generic
    financial commentary unrelated to the calculation."""

CONR_USER_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}

[TABLE]
{table}

[TEXT AFTER TABLE]
{post_text}

### QUESTION:
{question}

### YOUR TASK:
Solve this independently -- you have not been given the program or the answer.
Write the <think>...</think> reasoning trace and <program> block as instructed."""

## Load the teacher model with vLLM

`tensor_parallel_size=2` splits the model across both GPUs. First run will
download the full model from Hugging Face (~160GB in BF16) -- this can take
15-30+ minutes depending on bandwidth, during which the GPUs are idle but still
billed. Consider starting this cell and doing something else while it downloads.

In [ ]:
import os

# RTX PRO 6000 (Blackwell, SM120/compute capability 12.0) hits two separate
# FlashInfer JIT-compile failures with this model, at two different stages:
#
# 1. MoE kernels: vLLM's default "auto" moe_backend picks FlashInfer CUTLASS,
#    whose codegen doesn't support SM120 -> "RuntimeError: No supported CUDA
#    architectures found for major versions [12]". Fixed by forcing
#    moe_backend="triton" below (a real vLLM EngineArgs field).
#
# 2. Sampler: vLLM defaults to FlashInfer for top-k/top-p sampling too. Its
#    arch-support check (flashinfer/jit/core.py: check_cuda_arch) doesn't
#    recognize SM120 as valid yet and raises "RuntimeError: FlashInfer
#    requires GPUs with sm75 or higher" -- misleading message (SM120 > SM75),
#    it's really "SM120 isn't in FlashInfer's known-good list yet".
#    VLLM_USE_FLASHINFER_SAMPLER is a real vLLM env var (see vllm/envs.py);
#    setting it to "0" forces the plain PyTorch-native top-k/top-p sampler
#    instead, sidestepping this JIT path entirely. Must be set before vllm
#    is imported (env vars are read at import/engine-init time).
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

from vllm import LLM, SamplingParams

MODEL_NAME = "Qwen/Qwen3-Next-80B-A3B-Thinking"

llm = LLM(
    model=MODEL_NAME,
    tensor_parallel_size=2,
    dtype="bfloat16",
    max_model_len=16384,   # generous headroom for long <think> traces
    # 0.95 (was 0.90): the trial run reported only 5.83 GiB left for the KV
    # cache, capping concurrency at ~27 full-length sequences even though
    # max_num_seqs is 800. Measured usage was 74.3 GiB weights + 3.02 GiB peak
    # activation + 0.1 GiB non-torch + 2.52 GiB CUDA graphs = 79.94 GiB, so a
    # 0.95 budget (90.2 GiB) leaves ~10.3 GiB for KV -- about 1.8x the
    # concurrency, which matters when the batch is 2993 samples x N_SAMPLES
    # rather than 100. vLLM itself suggested going as high as 14.04 GiB; 0.95
    # keeps ~4.8 GiB of headroom instead of running the card to the edge.
    gpu_memory_utilization=0.95,
    trust_remote_code=True,
    moe_backend="triton",
    # Qwen3-Next's hybrid attention uses a separate Mamba cache block per
    # concurrently-running sequence (unlike plain attention KV cache, which
    # can be shared/paged more flexibly). vLLM's default max_num_seqs=1024
    # exceeded what fits in the ~5.4GiB of KV cache memory left over after
    # loading the 74.3GiB/GPU model weights, which only leaves room for 861
    # Mamba cache blocks -- vLLM refuses to start rather than silently running
    # with fewer concurrent sequences than requested:
    #   ValueError: max_num_seqs (1024) exceeds available Mamba cache blocks (861)
    # Capped below the observed 861 limit with some headroom. If you hit this
    # again with a different max_model_len/gpu_memory_utilization (which change
    # how much memory is left for the KV/Mamba cache), lower this further.
    max_num_seqs=800,
)

tokenizer = llm.get_tokenizer()
print(f"Loaded {MODEL_NAME} across 2 GPUs.")

## Generate traces (batched via vLLM)

vLLM handles all requests as a single batch internally (continuous batching), so this is one
`llm.generate()` call rather than a per-sample loop.

`N_SAMPLES` controls how many independent attempts are generated per prompt in that same batched
call (`SamplingParams(n=...)`). The validation step below keeps the first attempt that matches gold,
so extra attempts convert samples that a single try would have lost -- at a roughly linear cost in
time. `max_tokens` is raised to 8192 because 9.2% of the v1 run never emitted a `<program>` at all,
having burned the 4096-token budget on English self-talk.

In [ ]:
prompts = []
for s in samples:
    user_msg = CONR_USER_FRAME.format(
        pre_text=s["pre_text"], table=s["table"], post_text=s["post_text"],
        question=s["question"],
    )
    messages = [
        {"role": "system", "content": CONR_SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompts.append(text)

# max_tokens=8192 (was 4096): in the v1 run 220/2993 samples produced no
# <program> block, having spent the whole budget second-guessing themselves.
# The style rules in the prompt should shorten most traces, but the extra
# headroom means a long derivation still finishes.
#
# N_SAMPLES: independent attempts per prompt, generated in this same batched
# call. 2 roughly doubles cost/time but recovers samples lost to a single bad
# roll; set to 1 for the cheapest possible run.
N_SAMPLES = 2

sampling_params = SamplingParams(
    temperature=0.6,
    top_p=0.9,
    max_tokens=8192,
    n=N_SAMPLES,
)

outputs = llm.generate(prompts, sampling_params)
print(f"Generated {len(outputs)} responses ({N_SAMPLES} attempt(s) each).")

## Parse + validate each trace

Same two checks as the CoNR notebook:
1. `<program>` block must EXACT-match the gold program (same operators/args/
   order/#N refs) via the existing `parse_program`/`compute_program_accuracy`
   logic used across the other notebooks -- not just execution-equivalent.
   This is now the real yield signal, since the teacher genuinely had to
   derive the program itself.
2. `<think>` block must not contain leak phrases -- kept as a defensive check
   even though nothing was revealed this time, in case the model hallucinates
   having been given an answer.

When `N_SAMPLES > 1`, each sample has multiple independent attempts; we keep
the first attempt that exact-matches gold (if any), else the first attempt
(for review), and record how many attempts it took.

In [ ]:
"""
Parser + evaluation utilities for FinQA/VLSP-2025 style computation programs.
(Same implementation as the 0-shot/1-shot/few-shot/SFT notebooks, with one fix:
parse_program now handles nested parentheses inside arguments, e.g. table column
names like "Bien LN HDSXKD (%)" or "P/E (x)" -- the original regex-based
`([a-zA-Z_]+)\\(([^()]*)\\)` couldn't match across a paren nested inside the
argument list, so any program referencing such a column raised ValueError and
got scored as a false mismatch even when generated == gold verbatim. Found via
manual review of the first 150-sample trial run: 9 of 12 "mismatches" were
actually this bug, not a real generation difference.)
"""

from typing import List, Tuple, Optional

VALID_OPERATORS = {
    "add", "subtract", "multiply", "divide", "exp", "greater",
    "table_sum", "table_average", "table_max", "table_min",
}
_NUM_RE = re.compile(r"^-?\d+(\.\d+)?$")
_REF_RE = re.compile(r"^#(\d+)$")


def _split_top_level(s: str, sep: str = ",") -> List[str]:
    parts, depth, current = [], 0, []
    for ch in s:
        if ch == "(":
            depth += 1; current.append(ch)
        elif ch == ")":
            depth -= 1; current.append(ch)
        elif ch == sep and depth == 0:
            parts.append("".join(current)); current = []
        else:
            current.append(ch)
    if current:
        parts.append("".join(current))
    return [p.strip() for p in parts if p.strip() != ""]


def parse_program(program_str: str) -> List[Tuple[str, List[str]]]:
    program_str = program_str.strip().rstrip(",").strip()
    if not program_str:
        raise ValueError("Empty program string.")

    steps: List[Tuple[str, List[str]]] = []
    name_pattern = re.compile(r"\s*([a-zA-Z_]+)\(")
    pos = 0
    text = program_str
    while pos < len(text):
        m = name_pattern.match(text, pos)
        if not m:
            raise ValueError(f"Malformed program near: '{text[pos:pos+30]}...'")
        op = m.group(1).strip()
        if op not in VALID_OPERATORS:
            raise ValueError(f"Unknown operator: '{op}'")

        # Find the matching close paren by depth-counting, so parens nested
        # inside an argument (e.g. a column name like "ROE (%)") don't
        # prematurely end the call.
        start = m.end() - 1  # index of the opening '('
        depth = 0
        i = start
        while i < len(text):
            if text[i] == "(":
                depth += 1
            elif text[i] == ")":
                depth -= 1
                if depth == 0:
                    break
            i += 1
        else:
            raise ValueError(f"Unbalanced parens near: '{text[start:start+30]}...'")

        args = _split_top_level(text[start + 1:i], sep=",")
        steps.append((op, args))
        pos = i + 1
        if pos < len(text) and text[pos] == ",":
            pos += 1

    if not steps:
        raise ValueError("No valid steps parsed.")
    return steps


def _normalize_program(program_str: str) -> List[Tuple[str, Tuple[str, ...]]]:
    steps = parse_program(program_str)
    normalized = []
    for op, args in steps:
        norm_args = []
        for a in args:
            a = a.strip()
            if _REF_RE.match(a) or a.lower() == "none":
                norm_args.append(a.lower())
            else:
                # A trailing '%' means the value IS a percentage, so it must be
                # divided by 100 -- merely stripping the sign treats "15.1%" as
                # 15.1, which is 100x off and silently accepts a teacher program
                # that the repo's official PA scorer would reject. (Measured on
                # the valid split: 2 samples slipped through this way, e.g. gold
                # subtract(15.1%, 15.7%) vs teacher subtract(15.1, 15.7).)
                # This now matches _parse_numeric_literal in the eval notebooks.
                cleaned = a.replace(",", "").replace("$", "").strip()
                is_percent = cleaned.endswith("%")
                if is_percent:
                    cleaned = cleaned[:-1].strip()
                try:
                    value = float(cleaned)
                    if is_percent:
                        value /= 100.0
                    norm_args.append(f"{round(value, 6)}")
                except ValueError:
                    norm_args.append(a)
        normalized.append((op, tuple(norm_args)))
    return normalized


def is_exact_program_match(generated_program: str, gold_program: str) -> bool:
    """Exact structural match (not just execution-equivalent) -- if the teacher
    reformatted/simplified the gold program, we want to know and discard it."""
    try:
        return _normalize_program(generated_program) == _normalize_program(gold_program)
    except ValueError:
        return False


# Regex patterns (not plain substrings) so common, legitimate phrasing like
# "van ban cho biet ..." / "văn bản cho biết ..." (citing what the source
# document says) doesn't false-positive. The first 150-sample trial run used
# plain substrings ("cho biết", "được cho") and flagged 6 samples, of which
# only 4 were real leaks -- 2 were the model citing pre_text/table content,
# which is expected and fine. These patterns require "chương trình"/"kết
# quả"/"đáp án" immediately followed by "được cho" to actually catch the
# forbidden pattern (the teacher referring to the program/answer itself as
# something it was handed), not just any use of "cho biết"/"được cho".
LEAK_PATTERNS = [
    r"chương trình\s+được cho",
    r"kết quả\s+được cho",
    r"đáp án\s+được cho",
    r"theo chương trình được cho",
    r"đã biết trước",
    r"\bverified\b",
    r"\bas given\b",
    r"\bwe are told\b",
    r"\bgiven answer\b",
    r"\bgiven program\b",
]


def find_leak_phrases(think_block: str) -> List[str]:
    lowered = think_block.lower()
    return [pat for pat in LEAK_PATTERNS if re.search(pat, lowered)]


def extract_think_and_program(raw_text: str):
    """
    Extract the reasoning trace and program string from the model's raw output.

    Qwen3-Next-Thinking doesn't emit a literal opening `<think>` tag -- the
    chat template already puts the model in "thinking" mode at the start of
    the assistant turn, so the decoded text starts directly with the
    reasoning content and only closes with `</think>`. The first 150-sample
    trial run showed this is 100% consistent (0/150 had an opening tag), which
    made every `think` come back None under the original `<think>(.*?)</think>`
    regex. So: treat everything before the first `</think>` as the reasoning
    block (whether or not an opening tag happens to be present), and extract
    `<program>...</program>` as before.
    """
    close_think_idx = raw_text.find("</think>")
    if close_think_idx == -1:
        think = None
    else:
        think = raw_text[:close_think_idx]
        think = re.sub(r"^\s*<think>\s*", "", think)  # strip an opening tag if present
        think = think.strip()

    program_match = re.search(r"<program>(.*?)</program>", raw_text, re.DOTALL)
    program = program_match.group(1).strip() if program_match else None

    return think, program

In [ ]:
results = []
n_exact_match = 0
n_missing_tags = 0
n_leak = 0

for s, output in zip(samples, outputs):
    chosen_raw = chosen_think = chosen_program = None
    chosen_exact = False
    attempts_used = 0
    # Try each independent attempt in order; stop at the first one that
    # genuinely matches gold. If none match, keep the first attempt for review.
    for attempt_idx, cand in enumerate(output.outputs, start=1):
        raw_text = cand.text
        think, program = extract_think_and_program(raw_text)
        exact_match = is_exact_program_match(program, s["program"]) if program else False
        if chosen_raw is None:
            chosen_raw, chosen_think, chosen_program, attempts_used = raw_text, think, program, attempt_idx
        if exact_match:
            chosen_raw, chosen_think, chosen_program, attempts_used = raw_text, think, program, attempt_idx
            chosen_exact = True
            break

    leaks = find_leak_phrases(chosen_think) if chosen_think else []

    if chosen_think is None or chosen_program is None:
        n_missing_tags += 1
    if chosen_exact:
        n_exact_match += 1
    if leaks:
        n_leak += 1

    results.append({
        "train_index": s["train_index"],
        "evidence_type": s["evidence_type"],
        "question": s["question"],
        "gold_program": s["program"],
        "gold_answer": s["answer"],
        "attempts_used": attempts_used,
        "raw_output": chosen_raw,
        "parsed_think": chosen_think,
        "parsed_program": chosen_program,
        "exact_program_match": chosen_exact,
        "leak_phrases_found": leaks,
    })

print(f"Total: {len(results)}")
print(f"Exact program match (independent solve): {n_exact_match} ({100 * n_exact_match / len(results):.1f}%)")
print(f"Missing <think>/<program> tags: {n_missing_tags}")
print(f"Traces with leak phrases: {n_leak}")

# --- did the v2 convention rules land? -----------------------------------
# v1 (no convention rules) scored 14.4% exact match on the full train split,
# and its misses were dominated by four fixable formatting habits. Counting
# them here says whether each rule is being obeyed, which is far more
# actionable than the headline yield alone on a small trial.
import re as _re

_V1_EXACT_RATE = 14.4
_checks = {
    "R1 nested call in args": lambda p: bool(_re.search(r"\((?:[^()]*)\b[a-zA-Z_]+\(", p)),
    "R2 trailing multiply(#N, 100)": lambda p: bool(_re.search(r"multiply\(#\d+,\s*100\)\s*$", p)),
    "R3 invented operator": lambda p: any(
        o not in VALID_OPERATORS for o in _re.findall(r"([a-zA-Z_]+)\(", p)),
    "R5 table_*() around one value": lambda p: bool(
        _re.search(r"table_(?:sum|average|max|min)\(\s*[^,()]+\s*\)", p)),
}
_progs = [r["parsed_program"] for r in results if r["parsed_program"]]
print(f"\nRule violations among {len(_progs)} parsed programs:")
for _name, _fn in _checks.items():
    _n = sum(1 for p in _progs if _fn(p))
    print(f"  {_name:32s} {_n:4d} ({100 * _n / max(len(_progs), 1):.1f}%)")

_rate = 100 * n_exact_match / len(results)
print(f"\nExact match {_rate:.1f}% vs v1 baseline {_V1_EXACT_RATE}% "
      f"-> {'BETTER' if _rate > _V1_EXACT_RATE else 'NOT better'}")
print("If the violation rates above are still high, fix the corresponding rule "
      "in CONR_SYSTEM_PROMPT before spending on the full split.")

## Save all results for review

Saves every sample's raw output + parsed fields + validation flags, regardless
of pass/fail, so the full set can be read through to judge trace quality and
yield before scaling `SAMPLE_LIMIT` up (or removing it) for a full run.

In [ ]:
# Named after the input split (e.g. train.json -> ..._train.json) so
# running this notebook twice (once per split) doesn't overwrite the other
# split's results.
output_path = OUTPUT_DIR / f"independent_solve_trace_{TRAIN_JSON_PATH.stem}.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump({
        "model": MODEL_NAME,
        "n_samples": len(results),
        "summary": {
            "exact_program_match": n_exact_match,
            "missing_tags": n_missing_tags,
            "leak_phrases": n_leak,
        },
        "results": results,
    }, f, ensure_ascii=False, indent=2)

print(f"Saved {len(results)} results to {output_path}")
print("Download this file from the Server Web UI file browser.")

In [ ]:
# Build the final training-ready file: same schema as
# datasets/ViNumQA/{train,valid}_with_reasoning_trace.json (pre_text/table/
# post_text/id/qa, with qa.reasoning_trace added) -- only samples where the
# teacher's independently-derived program exact-matched gold are kept.
final_samples = []
for r in results:
    if not r["exact_program_match"]:
        continue
    original = train_data[r["train_index"]]
    final_samples.append({
        "pre_text": original["pre_text"],
        "table": original["table"],
        "post_text": original["post_text"],
        "id": original["id"],
        "qa": {
            "question": original["qa"]["question"],
            "program": original["qa"]["program"],
            "exe_ans": original["qa"]["exe_ans"],
            "reasoning_trace": r["parsed_think"],
        },
    })

final_path = OUTPUT_DIR / f"{TRAIN_JSON_PATH.stem}_with_reasoning_trace.json"
with open(final_path, "w", encoding="utf-8") as f:
    json.dump(final_samples, f, ensure_ascii=False, indent=2)

print(f"Kept {len(final_samples)} / {len(results)} samples "
      f"({100 * len(final_samples) / len(results):.1f}%).")
print(f"Saved final training-ready file to {final_path}")
print(f"Download it and place it at datasets/ViNumQA/{final_path.name} locally.")